## 02. Ingest Customers Auto Loader — objective

- Follow incremental ingestion from discovery to persisted results.
- Separate file discovery, parsing and schema tracking.

Historical results; no code executed.
- [Original inputs: gizmobox/customers](../../../data/gizmobox/customers/) — 4 files.
- [Original inputs: gizmobox/customers_autoloader](../../../data/gizmobox/customers_autoloader/) — 4 files.

# 02. Ingest Customers Auto Loader

Read the original source and historical results; nothing was executed.

[Full original payload](../../../artifacts/notebook-payloads/dea03-etl-with-apache-spark-streaming/02.%20Ingest%20Customers%20Auto%20Loader.json)

## Stream Customers Data From Cloud Files to Delta Lake using Auto Loader
1. Read files from cloud storage using Auto Loader
1. Transform the dataframe to add the following columns
    -   file path: Cloud file path
    -   ingestion date: Current Timestamp
1. Write the transformed data stream to Delta Lake Table

### 1. Read files using Auto Loader

In [ ]:
customers_df = (
                    spark.readStream
                         .format("cloudFiles")
                         .option("cloudFiles.format", "json")
                         .option("cloudFiles.schemaLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_schema")
                         .option("cloudFiles.inferColumnTypes", "true")
                         .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")
                         .load("/Volumes/gizmobox/landing/operational_data/customers_autoloader/")
)

### 2. Transform the dataframe to add the following columns
- file path: Cloud file path
- ingestion date: Current Timestamp

In [ ]:
from pyspark.sql.functions import current_timestamp, col

customers_transformed_df = (
                                customers_df.withColumn("file_path", col("_metadata.file_path"))
                                            .withColumn("ingestion_date", current_timestamp())
)

### 3. Write the transformed data stream to Delta Table 

In [ ]:
streaming_query = (
                    customers_transformed_df.writeStream
                        .format("delta")
                        .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_checkpoint_stream")
                        .toTable("gizmobox.bronze.customers_autoloader")
)

In [ ]:
streaming_query.stop()

Archived Databricks source — shown for reading, not Python execution.

```sql
%sql
SELECT * FROM gizmobox.bronze.customers_autoloader;
```